In [9]:
'''Exercise 1 - Pandas filtering and transformation

Read the patients_processed.csv created on Day 8.
If not available, recreate patients.csv and load it.

1. Filter patients where fee is above the
   average fee. Do this in one line.
   Hint: df[df['fee'] > df['fee'].mean()]

2. Filter patients who are either from Delhi
   or Mumbai AND have fee above 300.

3. Create a new column called fee_after_discount:
   Senior patients (age >= 50) get 20% discount.
   All others pay full fee.
   Use apply() with a lambda or a named function.

4. Find the age group (Young/Adult/Senior from Day 8)
   that has the highest average fee.

5. Create a pivot-style summary using groupby():
   For each age_group show:
   - count of patients
   - mean fee
   - max fee
   - min fee
   Use groupby().agg() with a dict.

Concepts: boolean filtering, mean(), apply(),
          lambda, groupby().agg(), conditions'''

import pandas as pd


# 1. Patients where fee > average fee (one line)

import pandas as pd

df = pd.read_csv("patients_processed.csv")

# One-liner filter
above_avg_fee = df[df['fee'] > df['fee'].mean()]
import pandas as pd

df = pd.read_csv("patients_processed.csv")

# One-liner filter
above_avg_fee = df[df['fee'] > df['fee'].mean()]



# 2. Patients from Delhi or Mumbai AND fee > 300
delhi_mumbai = df[(df["city"].isin(["Delhi", "Mumbai"])) & (df["fee"] > 300)]
print("\n--- Delhi/Mumbai Patients with Fee > 300 ---")
print(delhi_mumbai)

# 3. New column fee_after_discount
df["fee_after_discount"] = df.apply(
    lambda row: row["fee"] * 0.8 if row["age"] >= 50 else row["fee"], axis=1
)
print("\n--- Fee After Discount ---")
print(df[["name", "age", "fee", "fee_after_discount"]])

# 4. Age group with highest average fee
avg_fee_by_age_group = df.groupby("age_group")["fee"].mean()
top_age_group = avg_fee_by_age_group.idxmax()
print("\n--- Age Group with Highest Average Fee ---")
print(f"{top_age_group} → {avg_fee_by_age_group[top_age_group]}")

# 5. Pivot-style summary with groupby().agg()
summary = df.groupby("age_group").agg(
    count=("patient_id", "count"),
    mean_fee=("fee", "mean"),
    max_fee=("fee", "max"),
    min_fee=("fee", "min")
)
print("\n--- Age Group Summary ---")
print(summary)




--- Delhi/Mumbai Patients with Fee > 300 ---
   patient_id       name  age blood_type   city  fee age_group
0           1  Raj Kumar   34         O+  Delhi  500     Adult

--- Fee After Discount ---
              name  age  fee  fee_after_discount
0        Raj Kumar   34  500               500.0
1      Sneha Patel   39  200               200.0
2     Tom Williams   46  750               750.0
3  Fatima Al-Sayed   29  600               600.0
4         Liu Yang   24  300               300.0
5     Ana Gonzalez   56  900               720.0
6     James Okafor   32  400               400.0
7     Emily Carter   19  150               150.0

--- Age Group with Highest Average Fee ---
Senior → 900.0

--- Age Group Summary ---
           count  mean_fee  max_fee  min_fee
age_group                                   
Adult          4     462.5      750      200
Senior         1     900.0      900      900
Young          3     350.0      600      150


In [3]:
'''Exercise 2 - Pandas: Reading and handling JSON

Create a file called hospital_config.json:

{
  "hospital_name": "City General Hospital",
  "departments": ["Cardiology", "Neurology",
                  "Orthopedics", "Dermatology",
                  "General Medicine"],
  "fee_structure": {
    "Cardiology": 800,
    "Neurology": 700,
    "Orthopedics": 650,
    "Dermatology": 400,
    "General Medicine": 250
  },
  "staff": [
    {"id": 1, "name": "Dr. Anil Mehta",
     "specialty": "Cardiology", "years": 15},
    {"id": 2, "name": "Dr. Sarah Johnson",
     "specialty": "Neurology", "years": 12},
    {"id": 3, "name": "Dr. Raj Singh",
     "specialty": "Orthopedics", "years": 8},
    {"id": 4, "name": "Dr. Lisa Park",
     "specialty": "Dermatology", "years": 6},
    {"id": 5, "name": "Dr. Carlos Torres",
     "specialty": "General Medicine", "years": 20}
  ]
}

Write code to:
1. Load the JSON using the json module.
2. Print the hospital name.
3. Convert the staff list into a pandas DataFrame.
4. Add a new column called standard_fee by mapping
   each doctor's specialty to fee_structure.
   Hint: df['specialty'].map(fee_structure_dict)
5. Find the doctor with the most experience.
6. Find the average standard_fee across all doctors.
7. Add a column called seniority:
   years >= 15 → "Principal"
   years >= 10 → "Senior"
   below 10    → "Associate"
8. Export the final DataFrame to doctors_info.csv.

Concepts: json.load(), pd.DataFrame(),
          map(), apply(), to_csv()


---------------------------------------------'''

import json

# Data for hospital_config.json
hospital_data = {
    "hospital_name": "City General Hospital",
    "departments": [
        "Cardiology", "Neurology", "Orthopedics",
        "Dermatology", "General Medicine"
    ],
    "fee_structure": {
        "Cardiology": 800,
        "Neurology": 700,
        "Orthopedics": 650,
        "Dermatology": 400,
        "General Medicine": 250
    },
    "staff": [
        {"id": 1, "name": "Dr. Anil Mehta", "specialty": "Cardiology", "years": 15},
        {"id": 2, "name": "Dr. Sarah Johnson", "specialty": "Neurology", "years": 12},
        {"id": 3, "name": "Dr. Raj Singh", "specialty": "Orthopedics", "years": 8},
        {"id": 4, "name": "Dr. Lisa Park", "specialty": "Dermatology", "years": 6},
        {"id": 5, "name": "Dr. Carlos Torres", "specialty": "General Medicine", "years": 20}
    ]
}

# Write to JSON file
with open("hospital_config.json", "w") as f:
    json.dump(hospital_data, f, indent=2)

print("hospital_config.json created successfully!")

import json
import pandas as pd

# 1. Load JSON file
with open("hospital_config.json", "r") as f:
    data = json.load(f)

# 2. Print hospital name
print(f"Hospital: {data['hospital_name']}")

# 3. Convert staff list into DataFrame
df = pd.DataFrame(data["staff"])
print("\n--- Staff DataFrame ---")
print(df)

# 4. Add standard_fee column by mapping specialty → fee_structure
fee_structure = data["fee_structure"]
df["standard_fee"] = df["specialty"].map(fee_structure)

# 5. Doctor with most experience
most_exp = df.loc[df["years"].idxmax()]
print(f"\nMost Experienced Doctor: {most_exp['name']} ({most_exp['years']} years)")

# 6. Average standard_fee across all doctors
avg_fee = df["standard_fee"].mean()
print(f"\nAverage Standard Fee: {avg_fee:.2f}")

# 7. Add seniority column
def seniority(years):
    if years >= 15:
        return "Principal"
    elif years >= 10:
        return "Senior"
    else:
        return "Associate"

df["seniority"] = df["years"].apply(seniority)

print("\n--- Final DataFrame ---")
print(df)

# 8. Export to CSV
df.to_csv("doctors_info.csv", index=False)
print("\ndoctors_info.csv written successfully!")

hospital_config.json created successfully!
Hospital: City General Hospital

--- Staff DataFrame ---
   id               name         specialty  years
0   1     Dr. Anil Mehta        Cardiology     15
1   2  Dr. Sarah Johnson         Neurology     12
2   3      Dr. Raj Singh       Orthopedics      8
3   4      Dr. Lisa Park       Dermatology      6
4   5  Dr. Carlos Torres  General Medicine     20

Most Experienced Doctor: Dr. Carlos Torres (20 years)

Average Standard Fee: 560.00

--- Final DataFrame ---
   id               name         specialty  years  standard_fee  seniority
0   1     Dr. Anil Mehta        Cardiology     15           800  Principal
1   2  Dr. Sarah Johnson         Neurology     12           700     Senior
2   3      Dr. Raj Singh       Orthopedics      8           650  Associate
3   4      Dr. Lisa Park       Dermatology      6           400  Associate
4   5  Dr. Carlos Torres  General Medicine     20           250  Principal

doctors_info.csv written successfully!


In [4]:
'''Exercise 3 - Pandas merge (introduction)

You have two DataFrames.
Create them from the CSV files or hardcode:

df_appointments (from appointments.csv)
df_doctors (from doctors.csv)

Task:
1. Merge appointments with doctors on doctor_id.
   Use pd.merge() with how='left'.

2. After merging, the DataFrame should have:
   appt_id, patient_id, doctor_id, appt_date,
   diagnosis, fee, doctor name, specialty,
   experience_years.

3. From the merged DataFrame:
   - Find which specialty generated
     the most total fee revenue.
   - Find the average fee per specialty.
   - Find which doctor had the most appointments.

4. Now merge appointments with patients
   on patient_id.
   Find the average fee paid per city.

Concepts: pd.merge(), how='left', on=,
          groupby(), sum(), mean(), idxmax()


---------------------------------------------
'''

import pandas as pd

# Example data (you can load from CSVs instead)
df_appointments = pd.DataFrame({
    "appt_id": [101, 102, 103, 104, 105],
    "patient_id": [1, 2, 3, 4, 5],
    "doctor_id": [1, 2, 1, 3, 2],
    "appt_date": ["2026-03-01", "2026-03-02", "2026-03-03", "2026-03-04", "2026-03-05"],
    "diagnosis": ["Cardiac Check", "Migraine", "Follow-up", "Fracture", "Seizure"],
    "fee": [800, 700, 800, 650, 700]
})

df_doctors = pd.DataFrame({
    "doctor_id": [1, 2, 3],
    "doctor_name": ["Dr. Anil Mehta", "Dr. Sarah Johnson", "Dr. Raj Singh"],
    "specialty": ["Cardiology", "Neurology", "Orthopedics"],
    "experience_years": [15, 12, 8]
})

df_patients = pd.DataFrame({
    "patient_id": [1, 2, 3, 4, 5],
    "name": ["Raj Kumar", "Sneha Patel", "Tom Williams", "Fatima Al-Sayed", "Liu Yang"],
    "city": ["Delhi", "Mumbai", "Chicago", "Dubai", "Beijing"]
})

# 1. Merge appointments with doctors
merged = pd.merge(df_appointments, df_doctors, on="doctor_id", how="left")

print("\n--- Merged Appointments + Doctors ---")
print(merged)

# 3a. Specialty with most total fee revenue
revenue_by_specialty = merged.groupby("specialty")["fee"].sum()
top_specialty = revenue_by_specialty.idxmax()
print(f"\nSpecialty with most revenue: {top_specialty} ({revenue_by_specialty[top_specialty]})")

# 3b. Average fee per specialty
avg_fee_specialty = merged.groupby("specialty")["fee"].mean()
print("\n--- Average Fee per Specialty ---")
print(avg_fee_specialty)

# 3c. Doctor with most appointments
appt_counts = merged["doctor_name"].value_counts()
top_doctor = appt_counts.idxmax()
print(f"\nDoctor with most appointments: {top_doctor} ({appt_counts[top_doctor]})")

# 4. Merge appointments with patients
merged_patients = pd.merge(df_appointments, df_patients, on="patient_id", how="left")

avg_fee_city = merged_patients.groupby("city")["fee"].mean()
print("\n--- Average Fee per City ---")
print(avg_fee_city)


--- Merged Appointments + Doctors ---
   appt_id  patient_id  doctor_id   appt_date      diagnosis  fee  \
0      101           1          1  2026-03-01  Cardiac Check  800   
1      102           2          2  2026-03-02       Migraine  700   
2      103           3          1  2026-03-03      Follow-up  800   
3      104           4          3  2026-03-04       Fracture  650   
4      105           5          2  2026-03-05        Seizure  700   

         doctor_name    specialty  experience_years  
0     Dr. Anil Mehta   Cardiology                15  
1  Dr. Sarah Johnson    Neurology                12  
2     Dr. Anil Mehta   Cardiology                15  
3      Dr. Raj Singh  Orthopedics                 8  
4  Dr. Sarah Johnson    Neurology                12  

Specialty with most revenue: Cardiology (1600)

--- Average Fee per Specialty ---
specialty
Cardiology     800.0
Neurology      700.0
Orthopedics    650.0
Name: fee, dtype: float64

Doctor with most appointments: Dr. Anil

In [5]:
'''Exercise 4 - *args and **kwargs revision
            with data engineering context

Write a function called generate_report that:
- Takes a title as a required argument
- Takes any number of data lines via *args
  (each line is a string to print)
- Takes optional formatting via **kwargs:
  separator (default "=")
  width (default 40)
  show_count (default True)

The function should:
- Print the title centered within width chars
- Print separator line of width chars
- Print each data line
- If show_count is True, print total line count

Test with:
generate_report(
    "Patient Summary",
    "Total Patients : 8",
    "Avg Fee        : 485",
    "Top City       : Madrid",
    separator="-",
    width=45,
    show_count=True
)

Concepts: *args, **kwargs, .center(),
          string multiplication, f-string


---------------------------------------------
'''
def generate_report(title, *args, **kwargs):
    # Defaults
    separator = kwargs.get("separator", "=")
    width = kwargs.get("width", 40)
    show_count = kwargs.get("show_count", True)

    # Title centered
    print(title.center(width))
    # Separator line
    print(separator * width)

    # Print each data line
    for line in args:
        print(line)

    # Show count if enabled
    if show_count:
        print(f"\nTotal lines: {len(args)}")


# ---- Test run ----
generate_report(
    "Patient Summary",
    "Total Patients : 8",
    "Avg Fee        : 485",
    "Top City       : Madrid",
    separator="-",
    width=45,
    show_count=True
)


               Patient Summary               
---------------------------------------------
Total Patients : 8
Avg Fee        : 485
Top City       : Madrid

Total lines: 3


In [6]:
'''Exercise 5 - Lambda and apply() revision

Given this DataFrame:

import pandas as pd
data = {
    "name":   ["Raj", "Sneha", "Tom",
               "Fatima", "Liu", "Ana"],
    "age":    [34, 39, 46, 29, 24, 56],
    "fee":    [500, 200, 750, 600, 300, 900],
    "city":   ["Delhi","Mumbai","Chicago",
               "Dubai","Beijing","Madrid"]
}
df = pd.DataFrame(data)

1. Use apply() with lambda to create a column
   fee_in_inr converting USD fee to INR
   (multiply by 83).

2. Use apply() with lambda to create a column
   name_length with length of each name.

3. Use apply() with a named function to create
   a column called risk_category:
   age > 50 AND fee > 700 → "High Risk"
   age > 40              → "Medium Risk"
   otherwise             → "Low Risk"

4. Use apply() on the entire row (axis=1)
   to create a column called profile_tag
   combining name and city:
   Example: "Raj | Delhi"

Concepts: apply(), lambda, axis=0, axis=1,
          named function inside apply

'''
import pandas as pd

# Sample DataFrame
data = {
    "name":   ["Raj", "Sneha", "Tom", "Fatima", "Liu", "Ana"],
    "age":    [34, 39, 46, 29, 24, 56],
    "fee":    [500, 200, 750, 600, 300, 900],
    "city":   ["Delhi","Mumbai","Chicago","Dubai","Beijing","Madrid"]
}
df = pd.DataFrame(data)

# 1. Fee in INR (USD → INR, multiply by 83)
df["fee_in_inr"] = df["fee"].apply(lambda x: x * 83)

# 2. Name length
df["name_length"] = df["name"].apply(lambda x: len(x))

# 3. Risk category (named function)
def risk_category(row):
    if row["age"] > 50 and row["fee"] > 700:
        return "High Risk"
    elif row["age"] > 40:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_category"] = df.apply(risk_category, axis=1)

# 4. Profile tag (row-wise apply)
df["profile_tag"] = df.apply(lambda row: f"{row['name']} | {row['city']}", axis=1)

print("\n--- Final DataFrame ---")
print(df)



--- Final DataFrame ---
     name  age  fee     city  fee_in_inr  name_length risk_category  \
0     Raj   34  500    Delhi       41500            3      Low Risk   
1   Sneha   39  200   Mumbai       16600            5      Low Risk   
2     Tom   46  750  Chicago       62250            3   Medium Risk   
3  Fatima   29  600    Dubai       49800            6      Low Risk   
4     Liu   24  300  Beijing       24900            3      Low Risk   
5     Ana   56  900   Madrid       74700            3     High Risk   

      profile_tag  
0     Raj | Delhi  
1  Sneha | Mumbai  
2   Tom | Chicago  
3  Fatima | Dubai  
4   Liu | Beijing  
5    Ana | Madrid  
